In [1]:
# %pip install Flask langchain langchain-google-genai chromadb pypdf python-dotenv
# %pip install -U langchain-community
# %pip install -q -U google-genai

### Step 2: Set Google API Key

In [2]:
import google.generativeai as genai
import os
from dotenv import load_dotenv # If using .env

# Make sure your API key is loaded first
load_dotenv() # If using .env
# os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") # Ensure it's set

# Configure the genai library directly
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("Error: GOOGLE_API_KEY not found in environment variables.")
else:
    genai.configure(api_key=api_key)
    print("Listing available models supporting 'embedContent':")
    try:
        found_embedding_model = False
        for m in genai.list_models():
            if 'embedContent' in m.supported_generation_methods:
                print(f"- Found: {m.name}")
                found_embedding_model = True
        if not found_embedding_model:
             print("No models supporting 'embedContent' found for this API key.")
             print("Consider checking your API key permissions or creating a new one.")

        print("\nListing other models (may not support embedContent):")
        for m in genai.list_models():
            if 'embedContent' not in m.supported_generation_methods:
                 print(f"- {m.name} (Supported methods: {m.supported_generation_methods})")

    except Exception as e:
        print(f"Error listing models: {e}")
        print("Please ensure your API key is correct and has access to the Gemini API.")

e:\Anaconda\envs\myweb\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Listing available models supporting 'embedContent':
- Found: models/embedding-001
- Found: models/text-embedding-004
- Found: models/gemini-embedding-exp-03-07
- Found: models/gemini-embedding-exp

Listing other models (may not support embedContent):
- models/embedding-gecko-001 (Supported methods: ['embedText', 'countTextTokens'])
- models/gemini-1.0-pro-vision-latest (Supported methods: ['generateContent', 'countTokens'])
- models/gemini-pro-vision (Supported methods: ['generateContent', 'countTokens'])
- models/gemini-1.5-pro-latest (Supported methods: ['generateContent', 'countTokens'])
- models/gemini-1.5-pro-001 (Supported methods: ['generateContent', 'countTokens', 'createCachedContent'])
- models/gemini-1.5-pro-002 (Supported methods: ['generateContent', 'countTokens', 'createCachedContent'])
- models/gemini-1.5-pro (Supported methods: ['generateContent', 'countTokens'])
- models/gemini-1.5-flash-latest (Supported methods: ['generateContent', 'countTokens'])
- models/gemini-1.5

In [3]:
import os
from dotenv import load_dotenv

load_dotenv() # Load variables from .env file
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found. Please set it in a .env file or as an environment variable.")

os.environ["GOOGLE_API_KEY"] = google_api_key

# Verify it's set (optional, be careful not to print the key)
print("GOOGLE_API_KEY is set.")

GOOGLE_API_KEY is set.


### Step 3: Initialize Google LLM and Embedding Model

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-preview-04-17")
embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

### Step 4: Set up RAG for Supporting Documents

In [5]:
# import sys
# print(sys.executable)
# !{sys.executable} -m pip install -U langchain-chroma
# !{sys.executable} -m pip install google-generativeai

In [6]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Define a directory to save the vector store
persis_directory = "./chroma_db"

# Check if the vector store already exists
if os.path.exists(persis_directory):
    print(f"Loading existing vector store from {persis_directory}")
    vector_store = Chroma(
        persist_directory = persis_directory,
        embedding_function = embedding_model
    )
else:
    print(f"Creating new vector store in {persis_directory}")
    file_path = "personal_statement_sample.pdf"

    try:
        loader = PyPDFLoader(file_path)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {file_path}")

        text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
        chunks = text_splitter.split_documents(documents)
        print(f"Split into {len(chunks)} chunks")

        # Create the vector store and persist it
        vector_store = Chroma.from_documents(
            documents = chunks,
            embedding = embedding_model,
            persist_directory = persis_directory
        )
        vector_store.persist()
        print(f"Vector store created and persisted in {persis_directory}")

    except FileNotFoundError:
        print(f"Error: Document file not found at {file_path}. RAG will not be available.")
        vector_store = None
    except Exception as e:
        print(f"An error occurred during vector store creation: {e}")
        vector_store = None

# Create a retriever
retriever = vector_store.as_retriever(search_kwargs = {"k" : 3}) if vector_store else None
if retriever:
    print("Retriever created.")
else:
    print("Retriever not created.")

Loading existing vector store from ./chroma_db
Retriever created.


### Step 5: Set Up Conversation Memory

In [7]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True)


C:\Users\user\AppData\Local\Temp\ipykernel_26424\3839391778.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True)


### Step 6: Design the AI Agent's Logic (Prompt)

In [8]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant specialized in helping users write personal statements.
Your goal is to guide the user, ask relevant questions about their experiences and goals, and suggest additions or edits to their personal statement draft.
Maintain a supportive and encouraging tone.

Here is the current draft of the personal statement:
{current_statement_draft}

Relevant supporting information (e.g., examples, tips):
{retrieved_context}

Based on the chat history, the user's input, and the current draft, provide:
1. A conversational response [CHAT].
2. A suggestion for how to update the personal statement draft [STATEMENT_SUGGESTION]. Format this part clearly.
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Note: The actual chaining/invocation logic will be within the Flask route handlers.

### Step 7: Set Up Flask Application

In [9]:
from flask import Flask, request, jsonify, render_template
import threading
import time # Needed for a brief pause
import logging # Optional: Configure logging for Flask

# Disable Flask's default request logging in notebook for cleaner output
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app = Flask(__name__)

# Simple in-memory storage for the personal statement draft and chat history
# This state will reset when the notebook kernel restarts.
# Persistence would require saving/loading to a file or DB.
personal_statement_draft = ""
chat_history_list = [] # For initial load and simple display rendering

# Define Flask routes
@app.route('/')
def index():
    return render_template('html_template.html')

@app.route('/chat', methods=['POST'])
def chat():
    global personal_statement_draft
    user_input = request.json.get('message')
    current_statement_draft_from_frontend = request.json.get('personal_statement_draft', '')
    personal_statement_draft = current_statement_draft_from_frontend # Update backend's state


    if not user_input:
        return jsonify({"error": "No message provided"}), 400

    # 1. Add user message to history (for display/memory)
    chat_history_list.append({"role": "user", "content": user_input})
    memory.save_context({"input": user_input}, {"output": ""})

    try:
        # 2. (Optional) Retrieve context from RAG
        retrieved_context = ""
        if retriever:
             retrieved_docs = retriever.invoke(user_input)
             retrieved_context = "\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Prepare input for the LLM
        chat_history_for_prompt = memory.load_memory_variables({})["chat_history"]

        messages_for_llm = [
            ("system", prompt.messages[0].prompt.template.format(
                current_statement_draft=personal_statement_draft,
                retrieved_context=retrieved_context
            )),
        ] + chat_history_for_prompt + [("human", user_input)]

        llm_response_object = llm.invoke(messages_for_llm)
        llm_response_text = llm_response_object.content

        # 4. Parse LLM output
        chat_response = "Sorry, I couldn't parse the response."
        statement_suggestion = ""

        chat_marker = "[CHAT]"
        statement_marker = "[STATEMENT_SUGGESTION]"

        if chat_marker in llm_response_text:
             parts = llm_response_text.split(chat_marker, 1)[1].split(statement_marker, 1)
             chat_response = parts[0].strip()
             if len(parts) > 1:
                 statement_suggestion = parts[1].strip()
        elif statement_marker in llm_response_text:
             statement_suggestion = llm_response_text.split(statement_marker, 1)[1].strip()
             chat_response = "Okay, I have a suggestion for the statement." # Default


        # 5. Add agent response to history
        chat_history_list.append({"role": "assistant", "content": chat_response})
        memory.save_context({"input": user_input}, {"output": chat_response})

        # 6. Return response
        return jsonify({
            "chat_response": chat_response,
            "statement_suggestion": statement_suggestion,
        })

    except Exception as e:
        print(f"Error during AI processing: {e}")
        import traceback
        traceback.print_exc()
        error_response = "An error occurred while processing your request."
        chat_history_list.append({"role": "assistant", "content": error_response})
        return jsonify({"chat_response": error_response}), 500

@app.route('/get_initial_state', methods=['GET'])
def get_initial_state():
     # This returns the state from the global variables, which reset on kernel restart.
     # For persistence, load from file/DB here.
     return jsonify({
         "personal_statement_draft": personal_statement_draft,
         "chat_history": chat_history_list
     })


# Function to run Flask app in a separate thread
# WARNING: Running Flask with app.run() inside a notebook cell can be unstable.
# This threading approach is a workaround for demos/testing.
# For more robust development, run the Flask app as a standard Python script
# in a separate terminal.
def run_flask():
    print("Starting Flask server...")
    # Use a specific port, e.g., 5000
    app.run(port=5000, debug=False, use_reloader=False) # debug=True can cause issues with threading

# Check if the Flask server is already running before starting a new thread
# (This is a basic check, might not be perfect)
try:
    threading.Thread(target=run_flask).start()
    # Give the server a moment to start
    time.sleep(2)
    print("Flask server should be running at http://127.0.0.1:5000/")
except Exception as e:
    print(f"Failed to start Flask server thread: {e}")

Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off
Flask server should be running at http://127.0.0.1:5000/
